# Titanic Survival — Full Logistic Regression Pipeline

## Project 5

### Objective
Build an end-to-end binary classification pipeline using **Logistic Regression** to predict
whether a Titanic passenger survived.

The project includes:
- EDA
- missing-value analysis
- feature engineering
- train/test split
- numerical imputation and scaling
- categorical imputation and one-hot encoding
- Logistic Regression
- classification metrics
- confusion matrix
- ROC-AUC and ROC curve
- 5-fold stratified cross-validation
- probability-threshold analysis
- coefficient and odds-ratio interpretation

### Dataset

This project uses the standard Titanic training dataset containing **891 passengers and
12 original columns**, including `Survived`, `Pclass`, `Sex`, `Age`, `SibSp`, `Parch`,
`Fare`, `Cabin`, and `Embarked`.

Kaggle's official Titanic competition provides `train.csv` with the survival target and
`test.csv` for the 418 passengers whose outcomes are not supplied. citeturn0search1turn0search11

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, log_loss, confusion_matrix, classification_report,
    roc_curve
)

BASE = Path.cwd()
if not (BASE / "dataset").exists():
    BASE = BASE.parent

DATA = BASE / "dataset" / "titanic_train.csv"
OUT = BASE / "outputs"
OUT.mkdir(exist_ok=True)

df = pd.read_csv(DATA)
display(df.head())
print("Shape:", df.shape)

## 1. Dataset Information

In [ ]:
display(df.info())
display(df.describe(include="all").T)

## 2. Missing Values

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing_report = pd.DataFrame({
    "Missing_Count": missing,
    "Missing_Percent": missing / len(df) * 100
})
display(missing_report)

## 3. Target Distribution

In [ ]:
display(df["Survived"].value_counts().rename_axis("Survived").reset_index(name="Count"))

plt.figure(figsize=(7,5))
df["Survived"].value_counts().sort_index().plot(kind="bar")
plt.title("Titanic Survival Distribution")
plt.xlabel("Survived")
plt.ylabel("Passengers")
plt.xticks([0,1], ["No","Yes"], rotation=0)
plt.show()

## 4. Survival by Sex and Passenger Class

In [ ]:
display(
    df.groupby("Sex")["Survived"].mean().reset_index(name="Survival_Rate")
)

display(
    df.groupby("Pclass")["Survived"].mean().reset_index(name="Survival_Rate")
)

pd.crosstab(df["Sex"], df["Survived"], normalize="index").plot(kind="bar")
plt.title("Survival Rate by Sex")
plt.ylabel("Proportion")
plt.xticks(rotation=0)
plt.show()

## 5. Feature Engineering

In [ ]:
def extract_title(name):
    title = name.split(",")[1].split(".")[0].strip()
    common = {"Mr":"Mr", "Miss":"Miss", "Mrs":"Mrs", "Master":"Master"}
    return common.get(title, "Rare")

work = df.copy()
work["Title"] = work["Name"].apply(extract_title)
work["FamilySize"] = work["SibSp"] + work["Parch"] + 1
work["IsAlone"] = (work["FamilySize"] == 1).astype(int)
work["FarePerPerson"] = work["Fare"] / work["FamilySize"]
work["HasCabin"] = work["Cabin"].notna().astype(int)

display(work[[
    "Name","Title","FamilySize","IsAlone","FarePerPerson","HasCabin","Survived"
]].head())

## 6. Select Modeling Features

`PassengerId`, `Name`, `Ticket`, and raw `Cabin` are excluded from direct modeling.
`Name` contributes useful information through the engineered `Title` feature, while
`Cabin` contributes through `HasCabin`.

This avoids letting high-cardinality identifiers and raw text dominate the model.

In [ ]:
feature_cols = [
    "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare",
    "Embarked", "Title", "FamilySize", "IsAlone",
    "FarePerPerson", "HasCabin"
]

X = work[feature_cols]
y = work["Survived"]

numeric_features = [
    "Pclass", "Age", "SibSp", "Parch", "Fare",
    "FamilySize", "IsAlone", "FarePerPerson", "HasCabin"
]
categorical_features = ["Sex", "Embarked", "Title"]

## 7. Preprocessing Pipeline

The preprocessing is fitted **only on the training data** through a scikit-learn
`Pipeline`/`ColumnTransformer`.

- Numeric: median imputation → StandardScaler
- Categorical: most-frequent imputation → OneHotEncoder
- `handle_unknown="ignore"` prevents unseen test categories from breaking inference.

In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", drop="first"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

## 8. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

## 9. Logistic Regression Model

In [ ]:
logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

logistic_model.fit(X_train, y_train)

y_prob = logistic_model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.50).astype(int)

## 10. Classification Metrics

In [ ]:
metrics = pd.DataFrame([{
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1": f1_score(y_test, y_pred),
    "ROC_AUC": roc_auc_score(y_test, y_prob),
    "Log_Loss": log_loss(y_test, y_prob)
}])

display(metrics)
metrics.to_csv(OUT / "logistic_regression_metrics.csv", index=False)

print(classification_report(y_test, y_pred))

## 11. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
display(pd.DataFrame(
    cm,
    index=["Actual_0", "Actual_1"],
    columns=["Predicted_0", "Predicted_1"]
))

plt.figure(figsize=(7,5))
plt.imshow(cm)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.xticks([0,1], ["Did not survive","Survived"])
plt.yticks([0,1], ["Did not survive","Survived"])
for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i,j], ha="center", va="center")
plt.colorbar()
plt.show()

## 12. ROC-AUC

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)

plt.figure(figsize=(7,5))
plt.plot(fpr, tpr, label=f"Logistic Regression (AUC = {auc:.3f})")
plt.plot([0,1], [0,1], "--")
plt.title("ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.show()

## 13. 5-Fold Stratified Cross-Validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_result = cross_validate(
    logistic_model,
    X, y,
    cv=cv,
    scoring=["accuracy", "precision", "recall", "f1", "roc_auc"],
    n_jobs=-1
)

cv_summary = pd.DataFrame({
    "Metric": ["Accuracy","Precision","Recall","F1","ROC_AUC"],
    "Mean": [
        cv_result["test_accuracy"].mean(),
        cv_result["test_precision"].mean(),
        cv_result["test_recall"].mean(),
        cv_result["test_f1"].mean(),
        cv_result["test_roc_auc"].mean()
    ],
    "Std": [
        cv_result["test_accuracy"].std(),
        cv_result["test_precision"].std(),
        cv_result["test_recall"].std(),
        cv_result["test_f1"].std(),
        cv_result["test_roc_auc"].std()
    ]
})

display(cv_summary)
cv_summary.to_csv(OUT / "cross_validation_results.csv", index=False)

## 14. Probability Threshold Analysis

In [ ]:
thresholds_to_test = np.arange(0.10, 0.91, 0.05)
rows = []

for threshold in thresholds_to_test:
    pred = (y_prob >= threshold).astype(int)
    rows.append({
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0)
    })

threshold_df = pd.DataFrame(rows)
display(threshold_df)
threshold_df.to_csv(OUT / "threshold_analysis.csv", index=False)

threshold_df.plot(
    x="Threshold",
    y=["Precision","Recall","F1"],
    marker="o",
    figsize=(8,5)
)
plt.title("Metrics vs Probability Threshold")
plt.ylabel("Score")
plt.show()

## 15. Logistic Regression Coefficients and Odds Ratios

In [ ]:
feature_names = logistic_model.named_steps["preprocessor"].get_feature_names_out()
coefficients = logistic_model.named_steps["classifier"].coef_[0]

coef_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients,
    "Odds_Ratio": np.exp(coefficients)
})
coef_df["Absolute_Coefficient"] = coef_df["Coefficient"].abs()
coef_df = coef_df.sort_values("Absolute_Coefficient", ascending=False)

display(coef_df.head(20))
coef_df.to_csv(OUT / "logistic_coefficients_odds_ratios.csv", index=False)

### Interpretation

For Logistic Regression:

`odds_ratio = exp(coefficient)`

- Positive coefficient → higher predicted odds of survival.
- Negative coefficient → lower predicted odds of survival.
- Odds ratio > 1 → odds increase.
- Odds ratio < 1 → odds decrease.

Interpret coefficients in the context of the preprocessing and reference categories created by
one-hot encoding.

## 16. Test Predictions

In [ ]:
predictions = df.loc[X_test.index, ["PassengerId","Survived"]].copy()
predictions["Predicted_Survived"] = y_pred
predictions["Survival_Probability"] = y_prob
predictions["Correct"] = (
    predictions["Survived"] == predictions["Predicted_Survived"]
).astype(int)

display(predictions.sort_values("PassengerId").head(20))
predictions.sort_values("PassengerId").to_csv(
    OUT / "test_predictions.csv", index=False
)

## 17. Conclusion

The project implements a complete Logistic Regression classification workflow.

### Main pipeline
**Raw Titanic data → EDA → feature engineering → train/test split → imputation →
scaling/encoding → Logistic Regression → probability prediction → classification metrics →
confusion matrix → ROC-AUC → cross-validation → coefficient interpretation**

The exported CSV files contain the final metrics, predictions, confusion matrix,
cross-validation results, threshold analysis, and coefficient/odds-ratio analysis.